# **Limpieza y construcción de escenarios (día + hora)**

## QUÉ HACE ESTE SCRIPT
---------------------
1. Audita el CSV crudo y reporta inconsistencias.
2. Decide qué filas se usan para construir el patrón histórico de tránsito.
3. Unifica el nombre de cada avenida a un identificador único y confiable (Por el problema que existia).
4. Calcula un índice de movilidad a partir del cociente entre la velocidad
   observada y la velocidad de referencia (current_speed_kmh /
   free_speed_kmh).

**Interpretación**

Valores cercanos a 1:
* movilidad cercana al flujo libre
* menor permanencia esperada de personas en los edificios
* mayor vulnerabilidad del componente de movilidad

Valores cercanos a 0:
* movilidad congestionada
* mayor presencia de personas en la vía pública
* menor vulnerabilidad

   En esta fase NO se invierte el índice.La incorporación de este componente dentro del índice global de riesgose realiza posteriormente durante la construcción del modelo de riesgo dinámico.
5. Agrega los datos por escenario "día de la semana + hora del día" (168 combinaciones posibles: 7 días x 24 horas).
6. Exporta:
   - tabla_ancha_avenida_x_escenario.csv  -> el entregable principal
     (filas = avenida, columnas = escenario, valores = índice medio)
   - tabla_larga_escenarios.csv           -> misma info en formato "tidy"
     (una fila por avenida-escenario), con media, desviación estándar y n
   - conteos_por_escenario.csv            -> cuántas observaciones sostienen
     cada celda (para saber en qué celdas confiar)
   - reporte_auditoria.txt                -> bitácora de todas las
     decisiones de limpieza tomadas, para citar en la metodología de tesis


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# CONFIGURACIÓN
RUTA_ENTRADA = r"C:\Users\Evelyn\Downloads\Datos Riesgo Sismos incial\trafico_normalizado.csv"
CARPETA_SALIDA = Path(r"C:\Users\Evelyn\Downloads\Datos Riesgo Sismos incial\salidas")
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

# Umbral mínimo de observaciones para considerar "confiable" el promedio de una celda avenida x escenario.
UMBRAL_N_MINIMO = 3

DIAS_ES = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]

# Reporte de auditoría: lista de strings que se va llenando durante todo el script y se guarda al final como bitácora de decisiones metodológicas.
bitacora = []

def log(mensaje, mostrar=True):
    """Agrega una línea a la bitácora de auditoría y opcionalmente la imprime en consola. Centralizar esto en una función evita que se nos
    olvide documentar alguna decisión."""
    bitacora.append(mensaje)
    if mostrar:
        print(mensaje)

In [2]:
# PASO 1 - CARGA Y AUDITORÍA INICIAL DEL CSV CRUDO
# Cargamos todo como texto en las columnas problemáticas (lat) para poder inspeccionar valores corruptos antes de forzar conversiones numéricas.
log("PASO 1 - AUDITORÍA DEL ARCHIVO CRUDO")

df_crudo = pd.read_csv(RUTA_ENTRADA, dtype={"lat": str, "lon": str})
log(f"Filas totales leídas: {len(df_crudo)}")
log(f"Columnas: {list(df_crudo.columns)}")

# 1a. Fuentes de datos presentes - aunque sabemos que es google y tomtom
log("\n Fuentes de datos (columna 'source') ")
conteo_fuentes = df_crudo["source"].value_counts()
log(conteo_fuentes.to_string())

# 1b. Nulos en columnas numéricas clave 
cols_numericas = ["travel_time_min", "traffic_delay_min", "distance_m", "current_speed_kmh", "free_speed_kmh"]
log("\n Valores nulos en columnas numéricas ")
log(df_crudo[cols_numericas].isna().sum().to_string())

# 1c. Validez del campo 'lat' 
# Detectamos filas donde 'lat' no se puede convertir a número tal cual viene en el archivo (por ejemplo, con una coma sobrante al final).
# Esto es una revisión DEFENSIVA: ya que como tal solo en la revision visual eran las de tomtom pero por decision las voy a quitar. Aunque no se si en las de google venga algo asi
def es_convertible_a_float(valor):
    try:
        float(valor)
        return True
    except (TypeError, ValueError):
        return False

df_crudo["lat_valida"] = df_crudo["lat"].apply(es_convertible_a_float)
filas_lat_invalida = df_crudo[~df_crudo["lat_valida"]]
log(f"\n Filas con 'lat' no numérico (formato corrupto): {len(filas_lat_invalida)} ")
if len(filas_lat_invalida) > 0:
    log(filas_lat_invalida[["street", "lat", "source", "timestamp"]].to_string(index=False))
    log("Decisión: estas filas se excluyen del cálculo (no se puede ubicar el punto en el espacio ni confiar en sus valores numéricos, que además vienen vacíos en este caso). " \
    "Se documenta aquí y NO se corrige el valor 'a mano' dentro del código, para no inventar una coordenada.")

# 1d. Coincidencia de nombres de avenida con street_qgis
# Como ya empareje el nombre manualmente con el otro codigo pero en otra columna ahora revisamos si ese emparejamiento es limpio (1 a 1) o si hay ambigüedades, 
# porque de eso depende si la unificación de nombres se puede automatizar con seguridad.
log("\n Revisión del campo 'street_qgis' ")
calles_sin_match = df_crudo.loc[df_crudo["street_qgis"].isna(), "street"].unique()
log(f"Avenidas SIN coincidencia en OSM (street_qgis nulo): {list(calles_sin_match)}")

mapeo = df_crudo.dropna(subset=["street_qgis"]).groupby("street")["street_qgis"].unique()
ambiguos = {k: list(v) for k, v in mapeo.items() if len(v) > 1}
log(f"Avenidas cuyo nombre corto mapea a MÁS DE UN nombre OSM distinto (ambigüedad): {ambiguos if ambiguos else 'ninguna'}")

if len(calles_sin_match) > 0:
    log("Decisión sobre 'street_qgis' nulo: NO se puede automatizar de forma segura. Renombrar por similitud de texto (fuzzy matching) es riesgoso porque el nombre correcto " \
    "podría confundirse con una avenida distinta que tenga un nombre parecido en la zona de estudio (por ejemplo, dos calles con la palabra 'Salamanca' o nombres muy similares" \
    "en la Roma/Condesa). Esto REQUIERE revisión manual en QGIS. En este script, esas filas se conservan con su nombre original ('street') y se marcan explícitamente como" \
    "'PENDIENTE_REVISION_MANUAL' ")
else:
    #Por revision visual este no va salir porque vi lo de slamaca
    log("No se detectaron avenidas sin coincidencia OSM.")

if not ambiguos:
    log("El mapeo street -> street_qgis es 1 a 1 para todas las avenidas con coincidencia. Esto significa que SÍ es seguro automatizar la unificación de nombres para esas " \
    "avenidas (no hay ambigüedad que resolver a mano).")

PASO 1 - AUDITORÍA DEL ARCHIVO CRUDO
Filas totales leídas: 10688
Columnas: ['street', 'lat', 'lon', 'timestamp', 'travel_time_min', 'traffic_delay_min', 'distance_m', 'current_speed_kmh', 'free_speed_kmh', 'source', 'year', 'hour', 'street_qgis']

 Fuentes de datos (columna 'source') 
source
google    10608
tomtom       80

 Valores nulos en columnas numéricas 
travel_time_min      4
traffic_delay_min    4
distance_m           4
current_speed_kmh    4
free_speed_kmh       4

 Filas con 'lat' no numérico (formato corrupto): 0 

 Revisión del campo 'street_qgis' 
Avenidas SIN coincidencia en OSM (street_qgis nulo): []
Avenidas cuyo nombre corto mapea a MÁS DE UN nombre OSM distinto (ambigüedad): {'Amsterdam': ['Avenida Amsterdam Cacahuamilpa', 'Avenida Amsterdam Popocatépetl']}
No se detectaron avenidas sin coincidencia OSM.


In [3]:
# PASO 2 - DECISIÓN SOBRE QUÉ FILAS ENTRAN AL PROMEDIO HISTÓRICO
log("PASO 2 - FILTRADO DE FUENTES")


# Justificación la fuente 'google' cubre ~19 días consecutivos (12-sep a 2-oct-2024),con mediciones cada ~30 minutos y cobertura de las 24 horas del día.
# La fuente 'tomtom' solo aporta 4 momentos puntuales, en un año distinto (2026) y sin cobertura horaria ni semanal. (que pues tecnicamnete fue porque 
# no me entere de la existencia d elos otrso datos hasta despues pero pues ya lo habia echo)
# Promediarlas juntas violaría el supuesto de que el promedio representa "el comportamiento típico": estaríamos mezclando una muestra grande y bien distribuida
# en el tiempo con una muestra minúscula y no representativa
fuente_valida = "google"
df = df_crudo[df_crudo["source"] == fuente_valida].copy()
n_excluidas = len(df_crudo) - len(df)
log(f"Se conservan solo filas con source == '{fuente_valida}'.")
log(f"Filas excluidas por fuente: {n_excluidas} "
    f"({df_crudo.loc[df_crudo['source'] != fuente_valida, 'source'].unique()})")
log(f"Filas restantes para el análisis histórico: {len(df)}")

# Como consecuencia de este filtro, las filas con 'lat' corrupta (que eran todas de tomtom) ya quedaron excluidas automáticamente. CONFIRMAS
quedan_lat_invalida = (~df["lat_valida"]).sum()
log(f"Filas con 'lat' inválida que quedan tras el filtro de fuente: {quedan_lat_invalida} "
    f"(se esperaba 0, ya que ese problema solo ocurría en la fuente excluida)")

# Verificación de nulos numéricos remanentes, ya solo dentro de 'google'.
nulos_restantes = df[cols_numericas].isna().sum().sum()
log(f"Nulos numéricos remanentes en la fuente 'google': {nulos_restantes}")
if nulos_restantes > 0:
    df = df.dropna(subset=cols_numericas)
    log("Se eliminaron esas filas (no se imputan valores de tránsito, porque inventar un valor de congestión introduciría un sesgo que después se propagaría al índice de riesgo).")


PASO 2 - FILTRADO DE FUENTES
Se conservan solo filas con source == 'google'.
Filas excluidas por fuente: 80 (<ArrowStringArray>
['tomtom']
Length: 1, dtype: str)
Filas restantes para el análisis histórico: 10608
Filas con 'lat' inválida que quedan tras el filtro de fuente: 0 (se esperaba 0, ya que ese problema solo ocurría en la fuente excluida)
Nulos numéricos remanentes en la fuente 'google': 0


In [4]:
# PASO 3 - UNIFICACIÓN DEL NOMBRE DE AVENIDA
log("PASO 3 - UNIFICACIÓN DE NOMBRES DE AVENIDA")

# Regla de unificación si existe 'street_qgis', ese es el nombre canónico porque ya está validado Si no existe, se conserva el nombre original y se marca como pendiente de
# revisión manual, en vez de descartarlo o adivinarlo.
def nombre_canonico(row):
    if pd.notna(row["street_qgis"]):
        return row["street_qgis"]
    else:
        return f"{row['street']} [PENDIENTE_REVISION_MANUAL]"

df["avenida"] = df.apply(nombre_canonico, axis=1)

avenidas_pendientes = sorted(a for a in df["avenida"].unique() if "PENDIENTE_REVISION_MANUAL" in a)
log(f"Avenidas con nombre canónico asignado automáticamente: "
    f"{df['avenida'].nunique() - len(avenidas_pendientes)}")
log(f"Avenidas marcadas para revisión manual: {avenidas_pendientes}")

# Caso especial: "Amsterdam" tiene DOS puntos de medición distintos (dos coordenadas lat/lon distintas) bajo el mismo nombre de avenida.
# Esto es reflejo de que es una avenida en herradura con dos tramos/sentidos. Los dos puntos de Amsterdam se promedian JUNTOS bajo una sola fila "Avenida Amsterdam".
# Para no perder la información de que existen dos tramos, se exporta además una tabla de granularidad fina (por punto) que podrá usarse en QGIS.
n_puntos_por_avenida = df.groupby("avenida")[["lat", "lon"]].apply(
    lambda x: x.drop_duplicates().shape[0]
)
avenidas_multipunto = n_puntos_por_avenida[n_puntos_por_avenida > 1]
log(f"\nAvenidas con más de un punto de medición distinto: "
    f"{dict(avenidas_multipunto)}")
if len(avenidas_multipunto) > 0:
    log("Decisión: se promedian junto con el resto de la avenida en la tabla principal (una fila por avenida), pero esto DILUYE cualquier diferencia real entre los dos tramos. " \
    "Se guarda una tabla adicional a nivel de punto para no perder esa información (espacial)." )

PASO 3 - UNIFICACIÓN DE NOMBRES DE AVENIDA
Avenidas con nombre canónico asignado automáticamente: 13
Avenidas marcadas para revisión manual: []

Avenidas con más de un punto de medición distinto: {}


In [5]:
#PASO 4 - PARSEO DE FECHA/HORA
log("PASO 4 - PARSEO DE TIMESTAMP")

# El timestamp original viene en un formato no estándar: "2024:09-12T19:20:08.905383" (usa ':' entre año y mes, en vez de '-'). pandas no lo reconoce
# directamente, así que normalizamos el separador antes de parsear. 
def parsear_timestamp(valor_original):
    # Reemplazamos SOLO la primera ':' (la que separa año de mes), dejando intactos los ':' de la hora (HH:MM:SS).
    valor_normalizado = valor_original.replace(":", "-", 1)
    return pd.to_datetime(valor_normalizado, format="%Y-%m-%dT%H:%M:%S.%f")

df["fecha_hora"] = df["timestamp"].apply(parsear_timestamp)

# Verificación: si alguna fila no pudo parsearse, se detiene el script en vez de seguir adelante con fechas nulas silenciosas.
n_fechas_invalidas = df["fecha_hora"].isna().sum()
log(f"Timestamps que no se pudieron interpretar: {n_fechas_invalidas}")
assert n_fechas_invalidas == 0, "Hay timestamps sin parsear: revisar formato antes de continuar."

log(f"Rango de fechas cubierto: {df['fecha_hora'].min()} a {df['fecha_hora'].max()}")
log(f"Días distintos cubiertos: {df['fecha_hora'].dt.date.nunique()}")

PASO 4 - PARSEO DE TIMESTAMP
Timestamps que no se pudieron interpretar: 0
Rango de fechas cubierto: 2024-09-12 19:20:08.905383 a 2024-10-02 10:51:06.688668
Días distintos cubiertos: 19


In [6]:
# PASO 5 - CONSTRUCCIÓN DEL ESCENARIO (DÍA DE LA SEMANA + HORA)
log("PASO 5 - CONSTRUCCIÓN DE ESCENARIOS DÍA + HORA")

# dt.weekday() devuelve 0=lunes ... 6=domingo
df["dia_semana"] = df["fecha_hora"].dt.weekday.map(lambda i: DIAS_ES[i])

# Hora del día como número entero 0-23. 
# Nota metodológica importante: las mediciones NO caen exactamente en la hora en punto así que agrupar por hora significa agrupar TODO lo que ocurrió entre 
# HH:00:00 y HH:59:59 dentro de la hora HH. 
df["hora_dia"] = df["fecha_hora"].dt.hour

df["escenario"] = df["dia_semana"] + "_" + df["hora_dia"].astype(str).str.zfill(2) + "h"

log("Ejemplo de escenarios generados:")
log(df[["street", "fecha_hora", "dia_semana", "hora_dia", "escenario"]].head(5).to_string(index=False))

# Cuántas semanas distintas aporta cada combinación día+hora, para saber qué tan robusto es "el promedio de los lunes a las 8am": si solo hay
# 2-3 lunes en la ventana de datos, el promedio de ese escenario descansa en muy pocas observaciones independientes.
semanas_por_escenario = df.groupby("escenario")["fecha_hora"].apply(
    lambda x: x.dt.isocalendar().week.nunique()
)
log(f"\nNúmero de semanas distintas que sostienen cada escenario "
    f"(mínimo / mediana / máximo): "
    f"{semanas_por_escenario.min()} / {semanas_por_escenario.median()} / {semanas_por_escenario.max()}")

PASO 5 - CONSTRUCCIÓN DE ESCENARIOS DÍA + HORA
Ejemplo de escenarios generados:
           street                 fecha_hora dia_semana  hora_dia  escenario
  Av. Insurgentes 2024-09-12 19:20:08.905383     Jueves        19 Jueves_19h
   Av. Nuevo León 2024-09-12 19:20:10.051706     Jueves        19 Jueves_19h
  Av. Chapultepec 2024-09-12 19:20:11.206146     Jueves        19 Jueves_19h
Cto. Bicentenario 2024-09-12 19:20:12.404759     Jueves        19 Jueves_19h
        Amsterdam 2024-09-12 19:20:13.560435     Jueves        19 Jueves_19h

Número de semanas distintas que sostienen cada escenario (mínimo / mediana / máximo): 2 / 2.0 / 3


In [7]:
# PASO 6 - ÍNDICE DE CONGESTIÓN NORMALIZADO
log("PASO 6 - CÁLCULO DEL ÍNDICE DE CONGESTIÓN")

# Por qué NO usamos 'travel_time_min' directamente: cada avenida tiene una longitud de tramo distinta ('distance_m' va de ~150 m hasta más de 5,600 m), así que un 
# travel_time_min alto puede significar "avenida muy larga y fluida" en vez de "avenida corta y muy congestionada". 

# En vez de eso, se construye un índice adimensional que sí es comparable entre avenidas de distinta longitud:
#   indice_congestion = current_speed_kmh / free_speed_kmh

# Interpretación:
"""Interpretación metodológica:Este índice representa únicamente el estado de la movilidad.

Valores cercanos a 1 indican que la velocidad observada es similar a la velocidad de referencia (flujo libre), lo que se interpreta como una
menor permanencia esperada de personas dentro de los edificios y, por tanto, una mayor vulnerabilidad asociada a la movilidad.

Valores cercanos a 0 representan condiciones de congestión, asociadas a una mayor permanencia esperada de personas en los edificios
y, en consecuencia, una menor vulnerabilidad.

Por esta razón el índice NO se invierte en esta fase. La relación entre este índice y el riesgo final se establece únicamente
durante la formulación del índice de riesgo dinámico (Fase 3)."""

df["indice_movilidad"] = df["current_speed_kmh"] / df["free_speed_kmh"]


log("Estadísticas descriptivas del índice de movilidad (current_speed_kmh / free_speed_kmh). "
    "Valores altos representan mejor movilidad y menor vulnerabilidad asociada a la movilidad.")
log(df["indice_movilidad"].describe().to_string())


PASO 6 - CÁLCULO DEL ÍNDICE DE CONGESTIÓN
Estadísticas descriptivas del índice de movilidad (current_speed_kmh / free_speed_kmh). Valores altos representan mejor movilidad y menor vulnerabilidad asociada a la movilidad.
count    10608.000000
mean         1.085511
std          0.265677
min          0.227273
25%          0.936527
50%          1.078431
75%          1.247788
max          2.021277


In [8]:
# PASO 7 - AGREGACIÓN: AVENIDA x ESCENARIO
log("PASO 7 - AGREGACIÓN POR AVENIDA Y ESCENARIO")

# Tabla "larga" (tidy): una fila por combinación avenida-escenario, con media, desviación estándar y número de observaciones. Se guarda en este formato porque es el más 
# flexible para revisiones posteriores o para volver a pivotear.
tabla_larga = (
    df.groupby(["avenida", "dia_semana", "hora_dia", "escenario"])["indice_movilidad"]
    .agg(media="mean", desviacion_estandar="std", n_observaciones="count")
    .reset_index()
)

log(f"Filas en la tabla larga (avenida x escenario): {len(tabla_larga)}")
log(f"Avenidas distintas: {tabla_larga['avenida'].nunique()}")
log(f"Escenarios distintos observados: {tabla_larga['escenario'].nunique()} "
    f"(máximo teórico: 7 días x 24 horas = 168)")

# Aviso de robustez: marcamos qué proporción de celdas tiene pocas observaciones. Esto NO se elimina de la tabla (la información sigue siendo el mejor estimado disponible), 
# pero se reporta explícitamente para que se use con cautela y se justifique el umbral.
tabla_larga["confiable"] = tabla_larga["n_observaciones"] >= UMBRAL_N_MINIMO
pct_confiable = 100 * tabla_larga["confiable"].mean()
log(f"\nCeldas avenida-escenario con al menos {UMBRAL_N_MINIMO} observaciones: "
    f"{pct_confiable:.1f}% del total")
log(f"Celdas con menos de {UMBRAL_N_MINIMO} observaciones: "
    f"{(~tabla_larga['confiable']).sum()} de {len(tabla_larga)}")

PASO 7 - AGREGACIÓN POR AVENIDA Y ESCENARIO
Filas en la tabla larga (avenida x escenario): 2184
Avenidas distintas: 13
Escenarios distintos observados: 168 (máximo teórico: 7 días x 24 horas = 168)

Celdas avenida-escenario con al menos 3 observaciones: 100.0% del total
Celdas con menos de 3 observaciones: 0 de 2184


In [9]:
# PASO 8 - TABLA ANCHA (AVENIDA x ESCENARIO) - ENTREGABLE PRINCIPAL
log("PASO 8 - CONSTRUCCIÓN DE LA TABLA ANCHA")

tabla_ancha = tabla_larga.pivot_table(index="avenida", columns="escenario", values="media")

# Ordenamos las columnas en orden cronológico natural (Lunes_00h ...Domingo_23h) en vez de dejarlas en orden alfabético, que es lo que hace pivot_table por defecto y sería 
# difícil de leer.
orden_columnas = [f"{dia}_{hora:02d}h" for dia in DIAS_ES for hora in range(24)]
orden_columnas_presentes = [c for c in orden_columnas if c in tabla_ancha.columns]
tabla_ancha = tabla_ancha[orden_columnas_presentes]

# Tabla paralela de conteos (mismo formato ancho), para poder cruzar visualmente "qué tan confiable es cada celda de la tabla de medias".
tabla_conteos = tabla_larga.pivot_table(
    index="avenida", columns="escenario", values="n_observaciones"
)[orden_columnas_presentes]

log(f"Dimensiones de la tabla ancha final: {tabla_ancha.shape[0]} avenidas x "
    f"{tabla_ancha.shape[1]} escenarios")
log(f"Celdas vacías (combinación avenida-escenario sin ninguna observación): "
    f"{tabla_ancha.isna().sum().sum()} de {tabla_ancha.size}")

PASO 8 - CONSTRUCCIÓN DE LA TABLA ANCHA
Dimensiones de la tabla ancha final: 13 avenidas x 168 escenarios
Celdas vacías (combinación avenida-escenario sin ninguna observación): 0 de 2184


In [10]:
# PASO 9 - EXPORTACIÓN DE RESULTADOS
log("PASO 9 - EXPORTACIÓN")

ruta_tabla_ancha = CARPETA_SALIDA / "tabla_ancha_avenida_x_escenario.csv"
ruta_tabla_larga = CARPETA_SALIDA / "tabla_larga_escenarios.csv"
ruta_conteos = CARPETA_SALIDA / "conteos_por_escenario.csv"
ruta_puntos_finos = CARPETA_SALIDA / "tabla_larga_por_punto_medicion.csv"
ruta_bitacora = CARPETA_SALIDA / "reporte_auditoria.txt"

tabla_ancha.to_csv(ruta_tabla_ancha, encoding="utf-8-sig")
tabla_larga.to_csv(ruta_tabla_larga, index=False, encoding="utf-8-sig")
tabla_conteos.to_csv(ruta_conteos, encoding="utf-8-sig")

# PASO 9.1 - GENERACIÓN DE TABLAS POR DÍA PARA QGIS (OPCIONAL)
""" Este paso crea una tabla independiente para cada día de la semana a partir de la tabla ancha (avenida × escenario).
Objetivo:
Facilitar el trabajo en QGIS durante la Fase 2, evitando utilizar una tabla con las 168 columnas simultáneamente.

Salida:
movilidad_Lunes.csv
movilidad_Martes.csv
...
movilidad_Domingo.csv

Cada archivo conserva una fila por avenida y únicamente las 24 columnas correspondientes al día seleccionado.
Este paso no modifica ningún resultado del análisis ni genera información nueva; únicamente reorganiza el formato de salida para facilitar el proceso de join espacial."""

tabla_ancha_export = tabla_ancha.reset_index()

dias = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]

for dia in dias:
    #Columnas a ese día
    columnas = [c for c in tabla_ancha_export.columns if c.startswith(dia)]

    salida = tabla_ancha_export[["avenida"] + columnas].copy()

    #Renombrar columnas, ej: Lunes_00h -> 00h
    salida.rename(columns={c: c.replace(f"{dia}_", "") for c in columnas}, inplace=True)

    ruta = CARPETA_SALIDA / f"movilidad_{dia}.csv"
    salida.to_csv( ruta, index=False, encoding="utf-8-sig")

log("\nArchivos auxiliares para QGIS:")
log(f"Ubicación: {CARPETA_SALIDA}")

for dia in dias:
    log(f"  - movilidad_{dia}.csv")

# Tabla auxiliar de granularidad fina (por punto de medición: lat/lon), pensada para el caso "Amsterdam", aunque lo corregi a mano mejor pero por si acaso
tabla_puntos = (df.groupby(["avenida", "lat", "lon", "dia_semana", "hora_dia", "escenario"])["indice_movilidad"].agg(media="mean", desviacion_estandar="std", 
                                                                                                                      n_observaciones="count").reset_index())
tabla_puntos.to_csv(ruta_puntos_finos, index=False, encoding="utf-8-sig")

with open(ruta_bitacora, "w", encoding="utf-8") as f:
    f.write("\n".join(bitacora))

log(f"\nArchivos generados en: {CARPETA_SALIDA}")
log(f"  - {ruta_tabla_ancha.name}  (entregable principal)")
log(f"  - {ruta_tabla_larga.name}")
log(f"  - {ruta_conteos.name}")
log(f"  - {ruta_puntos_finos.name}  (auxiliar, granularidad por punto)")
log(f"  - {ruta_bitacora.name}  (bitácora completa de decisiones)")


PASO 9 - EXPORTACIÓN

Archivos auxiliares para QGIS:
Ubicación: C:\Users\Evelyn\Downloads\Datos Riesgo Sismos incial\salidas
  - movilidad_Lunes.csv
  - movilidad_Martes.csv
  - movilidad_Miércoles.csv
  - movilidad_Jueves.csv
  - movilidad_Viernes.csv
  - movilidad_Sábado.csv
  - movilidad_Domingo.csv

Archivos generados en: C:\Users\Evelyn\Downloads\Datos Riesgo Sismos incial\salidas
  - tabla_ancha_avenida_x_escenario.csv  (entregable principal)
  - tabla_larga_escenarios.csv
  - conteos_por_escenario.csv
  - tabla_larga_por_punto_medicion.csv  (auxiliar, granularidad por punto)
  - reporte_auditoria.txt  (bitácora completa de decisiones)
